# Tiny GPT Reader: pretrain on English, then learn to answer questions

Trains a ~17.5M-parameter GPT that reads word pieces, on Simple English Wikipedia. Step 2 (later) teaches it to answer questions from a passage.

1. **Runtime > Change runtime type > T4 GPU**, then run the cells in order.
2. The smoke test takes about 2 minutes. The full run takes about 2 hours: keep this tab open.
3. Progress is saved to your Google Drive (`MyDrive/tiny-gpt-reader`) every few minutes. **If Colab disconnects**, run the cells again from the top: training continues where it stopped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Save progress to Google Drive
Colab asks for permission to use your Drive. The checkpoint, tokenizer and logs go to `MyDrive/tiny-gpt-reader`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/tiny-gpt-reader"

## Get the code

In [ ]:
import os
os.chdir("/content")
if os.path.exists("tiny-gpt"):
    !cd tiny-gpt && git pull --ff-only
else:
    !git clone --depth 1 https://github.com/Nenethsu/tiny-gpt.git
os.chdir("/content/tiny-gpt")
assert os.path.exists("pretrain.py"), "pretrain.py is missing: push the latest code to GitHub first"
!python bpe.py test

## Smoke test (about 2 minutes)
A tiny model for 60 steps, to catch problems before the long run. It downloads Wikipedia once (157 MB); the full run reuses it.

In [ ]:
!python pretrain.py --preset smoke --out /content/reader-smoke

## Full pretraining (about 2 hours)
Prints the loss, the time left and sample sentences every 500 steps. Loss should fall from about 9 to around 3.5 or lower.

If Colab disconnects: run the cells from the top again (mount Drive, get the code), then this one. It resumes from the last save.

In [ ]:
!python pretrain.py --out "$OUT"

## Results

In [ ]:
import json
log = json.load(open(f"{OUT}/training.json"))
print(f"{log['params']:,} parameters, {log['steps_done']:,} steps, {log['train_seconds'] / 60:.0f} min on {log['device']}")
print(f"{log['train_tokens']:,} training tokens from {log['articles']:,} articles")
print("final loss:", log["loss"][-1])
for s in log["samples"][-1]["texts"]:
    print("-", s[:200])

# Step 2: learn to answer questions (SQuAD 2.0)

Starts from the pretrained model above and fine-tunes it on about 130,000 questions about Wikipedia passages; a third have no answer in the passage, so it learns to say so. Saved to `MyDrive/tiny-gpt-reader/qa` and resumes the same way.

The exam is SQuAD 2.0's dev set: 11,873 questions it never trains on.

## Smoke test (about 2 minutes)
Makes a tiny pretrained model first (in case this is a new session), then fine-tunes it for 40 steps.

In [ ]:
!python pretrain.py --preset smoke --out /content/reader-smoke && python finetune.py --preset smoke --init /content/reader-smoke --out /content/reader-qa-smoke

## Fine-tune (about 30-45 minutes)
Prints exact match, F1, "no answer" accuracy and the safety-check numbers every 1,000 steps, then takes the full exam.

In [ ]:
!python finetune.py --init "$OUT" --out "$OUT/qa"

## Exam results

In [ ]:
import json
qa = json.load(open(f"{OUT}/qa/training.json"))
print(json.dumps(qa["final"], indent=1))
for e in qa["examples"]:
    print(f"Q: {e['question']}
   expected {e['expected'][0]!r}, got {e['got']!r}")